In [1]:
import os
import warnings

warnings.filterwarnings("ignore")

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import display, Markdown, clear_output

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    raise ValueError("Please set NVIDIA_API_KEY in your environment before running this notebook.")

# ── Model ────────────────────────────────────────────────────────────────────
client = ChatNVIDIA(
    model="z-ai/glm5",
    api_key=NVIDIA_API_KEY,
    temperature=1,
    top_p=1,
    max_completion_tokens=1000,
    model_kwargs={
        "chat_template_kwargs": {"enable_thinking": True, "clear_thinking": False}
    },
)

# ── Prompt Template ───────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a witty AI poet. Keep answers short, creative, and fun."),
        ("human", "{user_input}"),
    ]
)

# ── Chain ─────────────────────────────────────────────────────────────────────
chain = prompt | client

# ── Stream + Render ───────────────────────────────────────────────────────────
thinking, answer = "", ""

USER_QUERY = "how to not love someone?"

for chunk in chain.stream({"user_input": USER_QUERY}):
    if chunk.additional_kwargs and "reasoning_content" in chunk.additional_kwargs:
        thinking += chunk.additional_kwargs["reasoning_content"]
    answer += str(chunk.content)

    clear_output(wait=True)
    display(
        Markdown(
            f"""
### ⚙ SYSTEM
*You are a witty AI poet. Keep answers short, creative, and fun.*

---

**👤 User:** {USER_QUERY}

<details open>
<summary>🧠 Thinking</summary>

{thinking or "…"}
</details>

**🤖 AI:** 

{answer or "…"}
"""
        )
    )

KeyboardInterrupt: 